# Continuous Prithvi MAE pretraining data

This notebook streams annual Sentinel mosaics into sharded TFRecords for continued pretraining of Prithvi MAE. Each spatial patch/product is stored once with every available year in the configured range. The training pipeline then expands that record into rolling four-year windows such as `[2015, 2016, 2017, 2018]` through `[2021, 2022, 2023, 2024]`.

This representation avoids writing the same year into several overlapping four-year records. Images are stored as float16 and the TFRecord shards use GZIP compression. The model's reconstruction target is the original unmasked four-year tensor; random MAE masking belongs in the training transform and is intentionally not baked into these files.

The source has annual mosaics at the top of each product folder and another tiled representation in year subfolders. Only the top-level `YYYY.tif` mosaics are used, so every pixel is exported once per product.

## Pipeline

1. Discover annual mosaics for the four Sentinel products.
2. Audit years, bands, dimensions, CRS, transforms, and storage requirements.
3. Assign patches to train or validation by geographic block.
4. Read one full-width 224-row stripe at a time to suit the source TIFF block layout.
5. Write each available annual sequence once to compressed, sharded TFRecords.
6. Verify serialized examples and demonstrate on-the-fly consecutive four-year expansion.

The expensive source preview and export are guarded by `RUN_PREVIEW` and `RUN_EXPORT`.

In [2]:
import hashlib
import json
import math
import os
import shutil
import time
from collections import Counter
from contextlib import ExitStack, contextmanager
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import tensorflow as tf
from rasterio.windows import Window

# TensorFlow is needed only for TFRecord I/O here. Leave GPU RAM to the
# later PyTorch training process.
try:
    tf.config.set_visible_devices([], "GPU")
except RuntimeError:
    pass

print("TensorFlow:", tf.__version__)
print("Rasterio:", rasterio.__version__)
print("NumPy:", np.__version__)

TensorFlow: 2.10.1
Rasterio: 1.4.4
NumPy: 1.26.4


## Configuration

The requested year range ends at 2024, so the 2025 surface-reflectance mosaic is deliberately excluded. The harmonized products provide 2015-2024; both surface-reflectance products provide 2018-2024 and therefore produce four rolling windows each.

Float16 halves the raw image payload relative to float32. Export raises an error instead of clipping if a source value exceeds the float16 finite range. Existing outputs are protected unless `OVERWRITE_OUTPUTS=True`.

In [4]:
SOURCE_ROOT = Path(
    r"Y:\SPOT\2023\Asare\sentinel_aoi_all\original"
)
OUTPUT_ROOT = Path(r"D:\Data\ssm_temporal\prithvi_continous_pretraining")

VARIANTS = (
    "harmonized",
    "harmonized_cloudy",
    "sr_harmonized",
    "sr_harmonized_cloudy",
)
MIN_YEAR = 2015
MAX_YEAR = 2024
WINDOW_LENGTH = 4
DAY_OF_YEAR = 1  # Annual composites do not have one acquisition day.

PATCH_SIZE = 224
STRIDE = 224
BAND_INDEXES = (1, 2, 3, 4, 5, 6)
STORAGE_DTYPE_NAME = "float16"
STORAGE_DTYPE = np.dtype(STORAGE_DTYPE_NAME)
COMPRESSION = "GZIP"
SCHEMA_VERSION = "prithvi-mae-continuous-v1"

VAL_FRACTION = 0.20
SPLIT_SEED = 42
SPATIAL_BLOCK_DEGREES = 0.05
TRAIN_SHARDS = 64
VAL_SHARDS = 16

INCLUDE_EDGE_PATCHES = True
SKIP_WITHOUT_VALID_WINDOW = True
OVERWRITE_OUTPUTS = False
RUN_PREVIEW = False
RUN_EXPORT = False

# Development limits. Leave as None for the complete export.
MAX_VARIANTS = None
MAX_STRIPES_PER_VARIANT = None
MAX_PATCHES_PER_STRIPE = None
PROGRESS_EVERY = 500

PREVIEW_VARIANT = "harmonized"
PREVIEW_ROW = 0
PREVIEW_COL = 0
PREVIEW_START_YEAR = 2015

assert SOURCE_ROOT.exists(), SOURCE_ROOT
assert WINDOW_LENGTH == 4
assert STRIDE == PATCH_SIZE, "Stripe extraction assumes non-overlap"
assert STORAGE_DTYPE == np.dtype("float16")

print("Source:", SOURCE_ROOT)
print("Output:", OUTPUT_ROOT)
print("RUN_PREVIEW:", RUN_PREVIEW)
print("RUN_EXPORT:", RUN_EXPORT)

Source: Y:\SPOT\2023\Asare\sentinel_aoi_all\original
Output: D:\Data\ssm_temporal\prithvi_continous_pretraining
RUN_PREVIEW: False
RUN_EXPORT: False


## Discover annual sequences

Discovery is intentionally non-recursive. A source image must be named exactly `YYYY.tif` and sit directly inside one of the four configured product folders.

In [4]:
@dataclass(frozen=True)
class TemporalPlan:
    variant: str
    years: tuple
    paths: tuple


def consecutive_window_starts(years, length=WINDOW_LENGTH):
    years = np.asarray(years, dtype=np.int64)
    starts = []
    for start in range(max(0, len(years) - length + 1)):
        candidate = years[start : start + length]
        if np.all(np.diff(candidate) == 1):
            starts.append(start)
    return tuple(starts)


def discover_temporal_plans(source_root=SOURCE_ROOT):
    plans = []
    for variant in VARIANTS:
        folder = source_root / variant
        if not folder.is_dir():
            raise FileNotFoundError(folder)

        annual = {}
        excluded = []
        for path in sorted(folder.glob("*.tif")):
            if not (path.stem.isdigit() and len(path.stem) == 4):
                continue
            year = int(path.stem)
            if MIN_YEAR <= year <= MAX_YEAR:
                if year in annual:
                    raise ValueError(f"Duplicate year {year} in {folder}")
                annual[year] = path
            else:
                excluded.append(year)

        years = tuple(sorted(annual))
        if not consecutive_window_starts(years):
            raise ValueError(
                f"{variant} has no consecutive {WINDOW_LENGTH}-year window: {years}"
            )
        if excluded:
            print(f"{variant}: excluding years outside range: {excluded}")
        plans.append(
            TemporalPlan(
                variant=variant,
                years=years,
                paths=tuple(annual[year] for year in years),
            )
        )
    return plans


plans = discover_temporal_plans()
inventory = pd.DataFrame(
    [
        {
            "variant": plan.variant,
            "years": list(plan.years),
            "year_count": len(plan.years),
            "rolling_windows": [
                list(plan.years[start : start + WINDOW_LENGTH])
                for start in consecutive_window_starts(plan.years)
            ],
            "source_gb": sum(path.stat().st_size for path in plan.paths) / 1e9,
        }
        for plan in plans
    ]
)
display(inventory)

sr_harmonized: excluding years outside range: [2025]


,variant,years,year_count,rolling_windows,source_gb
0,harmonized,"[2015, 2016, 2017, 2018, 2019, 2020, 2021, 202...",10,"[[2015, 2016, 2017, 2018], [2016, 2017, 2018, ...",249.131198
1,harmonized_cloudy,"[2015, 2016, 2017, 2018, 2019, 2020, 2021, 202...",10,"[[2015, 2016, 2017, 2018], [2016, 2017, 2018, ...",249.131198
2,sr_harmonized,"[2018, 2019, 2020, 2021, 2022, 2023, 2024]",7,"[[2018, 2019, 2020, 2021], [2019, 2020, 2021, ...",174.391839
3,sr_harmonized_cloudy,"[2018, 2019, 2020, 2021, 2022, 2023, 2024]",7,"[[2018, 2019, 2020, 2021], [2019, 2020, 2021, ...",174.391839


## Source audit

All years and products must share one pixel grid so that a patch coordinate always describes the same ground area. The export is blocked if a file has fewer than six bands, a different shape, CRS, or transform, or an unsupported dtype.

In [5]:
def grid_signature(source):
    return (
        source.width,
        source.height,
        source.crs,
        tuple(source.transform),
    )


with rasterio.open(plans[0].paths[0]) as reference:
    REFERENCE_GRID = grid_signature(reference)
    REFERENCE_WIDTH = reference.width
    REFERENCE_HEIGHT = reference.height
    REFERENCE_TRANSFORM = reference.transform
    REFERENCE_CRS = reference.crs

audit_rows = []
audit_issues = []
for plan in plans:
    for year, path in zip(plan.years, plan.paths):
        with rasterio.open(path) as source:
            issues = []
            if source.count < len(BAND_INDEXES):
                issues.append(f"only {source.count} bands")
            if grid_signature(source) != REFERENCE_GRID:
                issues.append("grid differs from reference")
            if not all(np.issubdtype(np.dtype(dtype), np.number) for dtype in source.dtypes):
                issues.append(f"non-numeric dtype: {source.dtypes}")
            audit_rows.append(
                {
                    "variant": plan.variant,
                    "year": year,
                    "shape": (source.height, source.width),
                    "bands": source.count,
                    "dtype": source.dtypes[0],
                    "block_shape": source.block_shapes[0],
                    "size_gb": path.stat().st_size / 1e9,
                    "issues": issues,
                }
            )
            audit_issues.extend(
                f"{plan.variant}/{year}: {issue}" for issue in issues
            )

audit = pd.DataFrame(audit_rows)
display(audit)
print("Reference shape:", (REFERENCE_HEIGHT, REFERENCE_WIDTH))
print("Reference CRS:", REFERENCE_CRS)
print("Source files audited:", len(audit))
if audit_issues:
    print("AUDIT ISSUES:")
    for issue in audit_issues:
        print("  ", issue)
else:
    print("All source grids and band requirements passed.")

precision_rows = []
for plan in plans:
    path = plan.paths[len(plan.paths) // 2]
    with rasterio.open(path) as source:
        sample = source.read(
            indexes=BAND_INDEXES,
            window=Window(0, source.height // 2, source.width, 1),
            out_dtype="float32",
        )
    finite = sample[np.isfinite(sample)]
    quantized = finite.astype(STORAGE_DTYPE).astype(np.float32)
    error = np.abs(quantized - finite)
    precision_rows.append(
        {
            "variant": plan.variant,
            "sample_year": int(path.stem),
            "sample_min": float(finite.min()),
            "sample_max": float(finite.max()),
            "mean_abs_float16_error": float(error.mean()),
            "max_abs_float16_error": float(error.max()),
        }
    )
precision_audit = pd.DataFrame(precision_rows)
display(precision_audit)
if precision_audit["sample_max"].max() > np.finfo(np.float16).max:
    raise OverflowError("Sampled source values exceed float16 range")

,variant,year,shape,bands,dtype,block_shape,size_gb,issues
0,harmonized,2015,"(30374, 34175)",6,float32,"(1, 34175)",24.91312,[]
1,harmonized,2016,"(30374, 34175)",6,float32,"(1, 34175)",24.91312,[]
2,harmonized,2017,"(30374, 34175)",6,float32,"(1, 34175)",24.91312,[]
3,harmonized,2018,"(30374, 34175)",6,float32,"(1, 34175)",24.91312,[]
4,harmonized,2019,"(30374, 34175)",6,float32,"(1, 34175)",24.91312,[]
5,harmonized,2020,"(30374, 34175)",6,float32,"(1, 34175)",24.91312,[]
6,harmonized,2021,"(30374, 34175)",6,float32,"(1, 34175)",24.91312,[]
7,harmonized,2022,"(30374, 34175)",6,float32,"(1, 34175)",24.91312,[]
8,harmonized,2023,"(30374, 34175)",6,float32,"(1, 34175)",24.91312,[]
9,harmonized,2024,"(30374, 34175)",6,float32,"(1, 34175)",24.91312,[]


Reference shape: (30374, 34175)
Reference CRS: GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST]]
Source files audited: 34
All source grids and band requirements passed.


,variant,sample_year,sample_min,sample_max,mean_abs_float16_error,max_abs_float16_error
0,harmonized,2020,437.000000,3707.0,0.150818,1.0
1,harmonized_cloudy,2020,844.833313,4649.0,0.395622,2.0
2,sr_harmonized,2021,189.500000,8585.0,0.139356,2.0
3,sr_harmonized_cloudy,2021,924.000000,5400.0,0.354139,2.0


## Geographic split and size estimate

The split depends only on the patch-center geographic block. Product and year are excluded from the split key, so every representation of one location remains in the same partition. The estimate below is the float16 payload before GZIP and before invalid patches are skipped.

In [6]:
def iter_patch_offsets(width, height):
    for row in range(0, height, STRIDE):
        for col in range(0, width, STRIDE):
            patch_height = min(PATCH_SIZE, height - row)
            patch_width = min(PATCH_SIZE, width - col)
            if not INCLUDE_EDGE_PATCHES and (
                patch_height < PATCH_SIZE or patch_width < PATCH_SIZE
            ):
                continue
            yield row, col


def patch_center(transform, row, col, width, height):
    x = col + width / 2.0
    y = row + height / 2.0
    longitude, latitude = transform * (x, y)
    return float(latitude), float(longitude)


def spatial_block_key(latitude, longitude):
    return (
        math.floor(latitude / SPATIAL_BLOCK_DEGREES),
        math.floor(longitude / SPATIAL_BLOCK_DEGREES),
    )


def split_for_block(block):
    token = f"{SPLIT_SEED}|{block[0]}|{block[1]}".encode()
    value = int.from_bytes(hashlib.sha256(token).digest()[:8], "big")
    return "val" if value / float(2**64) < VAL_FRACTION else "train"


patch_offsets = list(iter_patch_offsets(REFERENCE_WIDTH, REFERENCE_HEIGHT))
split_counts = Counter()
split_blocks = {"train": set(), "val": set()}
for row, col in patch_offsets:
    height = min(PATCH_SIZE, REFERENCE_HEIGHT - row)
    width = min(PATCH_SIZE, REFERENCE_WIDTH - col)
    latitude, longitude = patch_center(
        REFERENCE_TRANSFORM, row, col, width, height
    )
    block = spatial_block_key(latitude, longitude)
    split = split_for_block(block)
    split_counts[split] += 1
    split_blocks[split].add(block)

assert split_blocks["train"].isdisjoint(split_blocks["val"])
patch_count = len(patch_offsets)
bytes_per_year_patch = (
    len(BAND_INDEXES) * PATCH_SIZE * PATCH_SIZE * STORAGE_DTYPE.itemsize
)
raw_payload_bytes = sum(
    patch_count * len(plan.years) * bytes_per_year_patch for plan in plans
)
rolling_examples = sum(
    patch_count * len(consecutive_window_starts(plan.years)) for plan in plans
)
max_stripe_bytes = max(
    len(BAND_INDEXES)
    * len(plan.years)
    * PATCH_SIZE
    * REFERENCE_WIDTH
    * STORAGE_DTYPE.itemsize
    for plan in plans
)

print("Patches per product:", f"{patch_count:,}")
print("Train/validation locations:", dict(split_counts))
print("Stored records before filtering:", f"{patch_count * len(plans):,}")
print("Rolling four-year examples before filtering:", f"{rolling_examples:,}")
print("Raw float16 payload before GZIP: {:.1f} GB".format(raw_payload_bytes / 1e9))
print("Maximum temporal stripe array: {:.2f} GB".format(max_stripe_bytes / 1e9))
output_drive = Path(OUTPUT_ROOT.anchor)
if output_drive.exists():
    free_bytes = shutil.disk_usage(output_drive).free
    print("Current output-drive free space: {:.1f} GB".format(free_bytes / 1e9))

Patches per product: 20,808
Train/validation locations: {'train': 16783, 'val': 4025}
Stored records before filtering: 83,232
Rolling four-year examples before filtering: 457,776
Raw float16 payload before GZIP: 426.0 GB
Maximum temporal stripe array: 0.92 GB
Current output-drive free space: 1169.9 GB


## Stripe-based patch extraction

These mosaics are organized as one-row TIFF blocks. Reading thousands of small horizontal windows can repeatedly decompress the same full rows. The export instead reads a full-width 224-row stripe for each year, converts it to float16, and slices all patch columns from memory. Peak array size is printed above. Edge patches are zero-padded, never resized.

In [7]:
@contextmanager
def open_plan_sources(plan):
    with ExitStack() as stack:
        sources = [
            stack.enter_context(rasterio.open(path)) for path in plan.paths
        ]
        for source in sources:
            if grid_signature(source) != REFERENCE_GRID:
                raise ValueError("Source grid changed after audit")
        yield sources


def read_temporal_stripe(sources, row):
    height = min(PATCH_SIZE, REFERENCE_HEIGHT - row)
    stripe = np.zeros(
        (
            len(BAND_INDEXES),
            len(sources),
            PATCH_SIZE,
            REFERENCE_WIDTH,
        ),
        dtype=STORAGE_DTYPE,
    )
    window = Window(0, row, REFERENCE_WIDTH, height)
    float16_limit = np.finfo(np.float16).max

    for timestep, source in enumerate(sources):
        data = source.read(
            indexes=BAND_INDEXES,
            window=window,
            out_dtype="float32",
            masked=False,
        )
        np.nan_to_num(data, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
        peak = float(np.max(np.abs(data))) if data.size else 0.0
        if peak > float16_limit:
            raise OverflowError(
                f"Value {peak} at row {row} exceeds float16 range"
            )
        stripe[:, timestep, :height, :] = data.astype(
            STORAGE_DTYPE, copy=False
        )
    return stripe


def make_patch_id(variant, row, col):
    text = f"{variant}|{row}|{col}|{SCHEMA_VERSION}"
    return hashlib.sha1(text.encode()).hexdigest()


def build_patch_record(plan, image, row, col):
    image = np.ascontiguousarray(image, dtype=STORAGE_DTYPE)
    temporal_mask = np.any(
        np.abs(image) > 0, axis=(0, 2, 3)
    ).astype(np.int64)
    year_starts = consecutive_window_starts(plan.years)
    valid_starts = [
        start
        for start in year_starts
        if temporal_mask[start : start + WINDOW_LENGTH].all()
    ]

    height = min(PATCH_SIZE, REFERENCE_HEIGHT - row)
    width = min(PATCH_SIZE, REFERENCE_WIDTH - col)
    latitude, longitude = patch_center(
        REFERENCE_TRANSFORM, row, col, width, height
    )
    block = spatial_block_key(latitude, longitude)
    split = split_for_block(block)
    patch_id = make_patch_id(plan.variant, row, col)

    return {
        "image": image,
        "years": np.asarray(plan.years, dtype=np.int64),
        "temporal_mask": temporal_mask,
        "location_coords": np.asarray(
            [latitude, longitude], dtype=np.float32
        ),
        "variant": plan.variant,
        "patch_id": patch_id,
        "patch_row": int(row),
        "patch_col": int(col),
        "spatial_block": f"{block[0]}:{block[1]}",
        "split": split,
        "valid_window_count": len(valid_starts),
    }


def extract_patch_from_stripe(plan, stripe, row, col):
    width = min(PATCH_SIZE, REFERENCE_WIDTH - col)
    if width == PATCH_SIZE:
        image = stripe[:, :, :, col : col + PATCH_SIZE]
    else:
        image = np.zeros(
            (len(BAND_INDEXES), len(plan.years), PATCH_SIZE, PATCH_SIZE),
            dtype=STORAGE_DTYPE,
        )
        image[:, :, :, :width] = stripe[:, :, :, col : col + width]
    return build_patch_record(plan, image, row, col)

## Optional source preview

This reads one patch directly from every year of the selected product and displays one valid four-year window. It is disabled by default because random small reads from these scanline TIFFs can still involve substantial source I/O.

In [8]:
def load_patch_direct(plan, row, col):
    height = min(PATCH_SIZE, REFERENCE_HEIGHT - row)
    width = min(PATCH_SIZE, REFERENCE_WIDTH - col)
    if height <= 0 or width <= 0:
        raise ValueError("Preview row/column is outside the raster")

    image = np.zeros(
        (len(BAND_INDEXES), len(plan.years), PATCH_SIZE, PATCH_SIZE),
        dtype=STORAGE_DTYPE,
    )
    window = Window(col, row, width, height)
    with open_plan_sources(plan) as sources:
        for timestep, source in enumerate(sources):
            data = source.read(
                indexes=BAND_INDEXES,
                window=window,
                out_dtype="float32",
                masked=False,
            )
            np.nan_to_num(data, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
            peak = float(np.max(np.abs(data))) if data.size else 0.0
            if peak > np.finfo(np.float16).max:
                raise OverflowError(f"Preview value {peak} exceeds float16 range")
            image[:, timestep, :height, :width] = data.astype(STORAGE_DTYPE)
    return build_patch_record(plan, image, row, col)


def frame_to_rgb(frame, divisor=3000.0):
    rgb = frame[[2, 1, 0]].transpose(1, 2, 0).astype(np.float32)
    return np.clip(rgb / divisor, 0.0, 1.0)


def visualize_four_year_window(record, requested_start_year=None):
    starts = [
        start
        for start in consecutive_window_starts(record["years"])
        if record["temporal_mask"][start : start + WINDOW_LENGTH].all()
    ]
    if not starts:
        raise ValueError("Patch has no valid consecutive four-year window")

    matching = [
        start for start in starts
        if int(record["years"][start]) == requested_start_year
    ]
    start = matching[0] if matching else starts[0]
    years = record["years"][start : start + WINDOW_LENGTH]

    figure, axes = plt.subplots(1, WINDOW_LENGTH, figsize=(16, 4))
    for offset, axis in enumerate(axes):
        frame = record["image"][:, start + offset]
        axis.imshow(frame_to_rgb(frame))
        axis.set_title(str(int(years[offset])))
        axis.axis("off")
    figure.suptitle(
        f"{record['variant']} | {record['split']} | {record['patch_id'][:10]}"
    )
    plt.tight_layout()
    plt.show()


if RUN_PREVIEW:
    preview_plan = next(plan for plan in plans if plan.variant == PREVIEW_VARIANT)
    preview_record = load_patch_direct(
        preview_plan, PREVIEW_ROW, PREVIEW_COL
    )
    print("Image shape:", preview_record["image"].shape)
    print("Years:", preview_record["years"].tolist())
    print("Temporal mask:", preview_record["temporal_mask"].tolist())
    print("Valid rolling windows:", preview_record["valid_window_count"])
    visualize_four_year_window(preview_record, PREVIEW_START_YEAR)
else:
    preview_record = None
    print("Source preview skipped because RUN_PREVIEW=False.")

Source preview skipped because RUN_PREVIEW=False.


## TFRecord schema

Each record contains one product/location sequence. `years` and `temporal_mask` are variable length because products have different temporal coverage. The image is always channel-first with shape `(6, year_count, 224, 224)`.

| Feature | Stored value |
|---|---|
| `image_raw` | float16 annual sequence bytes |
| `years` | ordered integer years |
| `temporal_mask` | 1 when that patch/year contains nonzero data |
| `location_coords` | patch-center `[latitude, longitude]` |
| `variant` | source product folder |
| `patch_id` | stable product/row/column identifier |
| `spatial_block` | deterministic geographic split group |
| `valid_window_count` | usable consecutive four-year windows |

There is no segmentation mask or separate reconstruction target. During training, `target = image` before random MAE masking is applied to the model input.

In [9]:
def bytes_feature(value):
    if isinstance(value, str):
        value = value.encode("utf-8")
    if isinstance(value, np.ndarray):
        value = np.ascontiguousarray(value).tobytes()
    return tf.train.Feature(bytes_list=tf.train.BytesList(value=[value]))


def int64_feature(value):
    if isinstance(value, np.ndarray):
        value = value.reshape(-1).tolist()
    elif not isinstance(value, (list, tuple)):
        value = [value]
    return tf.train.Feature(
        int64_list=tf.train.Int64List(value=[int(item) for item in value])
    )


def float_feature(value):
    if isinstance(value, np.ndarray):
        value = value.reshape(-1).tolist()
    elif not isinstance(value, (list, tuple)):
        value = [value]
    return tf.train.Feature(
        float_list=tf.train.FloatList(value=[float(item) for item in value])
    )


def serialize_patch(record):
    image = np.ascontiguousarray(record["image"], dtype=STORAGE_DTYPE)
    channels, timesteps, height, width = image.shape
    if channels != len(BAND_INDEXES) or (height, width) != (PATCH_SIZE, PATCH_SIZE):
        raise ValueError(f"Unexpected image shape: {image.shape}")
    if timesteps != len(record["years"]):
        raise ValueError("Image timestep count does not match years")
    if len(record["temporal_mask"]) != timesteps:
        raise ValueError("Temporal mask length does not match years")

    features = {
        "image_raw": bytes_feature(image),
        "height": int64_feature(height),
        "width": int64_feature(width),
        "channels": int64_feature(channels),
        "timesteps": int64_feature(timesteps),
        "years": int64_feature(record["years"]),
        "temporal_mask": int64_feature(record["temporal_mask"]),
        "location_coords": float_feature(record["location_coords"]),
        "variant": bytes_feature(record["variant"]),
        "patch_id": bytes_feature(record["patch_id"]),
        "patch_row": int64_feature(record["patch_row"]),
        "patch_col": int64_feature(record["patch_col"]),
        "spatial_block": bytes_feature(record["spatial_block"]),
        "split": bytes_feature(record["split"]),
        "valid_window_count": int64_feature(record["valid_window_count"]),
        "storage_dtype": bytes_feature(STORAGE_DTYPE_NAME),
        "schema_version": bytes_feature(SCHEMA_VERSION),
    }
    return tf.train.Example(
        features=tf.train.Features(feature=features)
    ).SerializeToString()


FEATURE_DESCRIPTION = {
    "image_raw": tf.io.FixedLenFeature([], tf.string),
    "height": tf.io.FixedLenFeature([], tf.int64),
    "width": tf.io.FixedLenFeature([], tf.int64),
    "channels": tf.io.FixedLenFeature([], tf.int64),
    "timesteps": tf.io.FixedLenFeature([], tf.int64),
    "years": tf.io.VarLenFeature(tf.int64),
    "temporal_mask": tf.io.VarLenFeature(tf.int64),
    "location_coords": tf.io.FixedLenFeature([2], tf.float32),
    "variant": tf.io.FixedLenFeature([], tf.string),
    "patch_id": tf.io.FixedLenFeature([], tf.string),
    "patch_row": tf.io.FixedLenFeature([], tf.int64),
    "patch_col": tf.io.FixedLenFeature([], tf.int64),
    "spatial_block": tf.io.FixedLenFeature([], tf.string),
    "split": tf.io.FixedLenFeature([], tf.string),
    "valid_window_count": tf.io.FixedLenFeature([], tf.int64),
    "storage_dtype": tf.io.FixedLenFeature([], tf.string),
    "schema_version": tf.io.FixedLenFeature([], tf.string),
}

## Training-time four-year expansion

The functions below are included now to lock down the preprocessing contract. The later training notebook can reuse them. `record_to_consecutive_windows` rejects year gaps and patch-level missing data, returns fixed `(6, 4, 224, 224)` tensors, and sets the unmasked image as the reconstruction target. Random spatial/temporal patch masking should be applied after this step.

In [7]:
def parse_pretraining_example(serialized):
    example = tf.io.parse_single_example(serialized, FEATURE_DESCRIPTION)
    timesteps = tf.cast(example["timesteps"], tf.int32)
    image = tf.io.decode_raw(example["image_raw"], tf.float16)
    image = tf.reshape(
        image,
        [len(BAND_INDEXES), timesteps, PATCH_SIZE, PATCH_SIZE],
    )
    image = tf.cast(image, tf.float32)
    image = tf.ensure_shape(
        image, [len(BAND_INDEXES), None, PATCH_SIZE, PATCH_SIZE]
    )
    years = tf.cast(tf.sparse.to_dense(example["years"]), tf.int32)
    temporal_mask = tf.cast(
        tf.sparse.to_dense(example["temporal_mask"]), tf.bool
    )
    return {
        "image": image,
        "years": years,
        "temporal_mask": temporal_mask,
        "location_coords": example["location_coords"],
        "variant": example["variant"],
        "patch_id": example["patch_id"],
    }


def record_to_consecutive_windows(record):
    timesteps = tf.shape(record["years"])[0]
    starts = tf.range(tf.maximum(timesteps - WINDOW_LENGTH + 1, 0))
    offsets = starts[:, tf.newaxis] + tf.range(WINDOW_LENGTH)[tf.newaxis, :]
    candidate_years = tf.gather(record["years"], offsets)
    candidate_masks = tf.gather(record["temporal_mask"], offsets)
    consecutive = tf.reduce_all(
        candidate_years[:, 1:] - candidate_years[:, :-1] == 1, axis=1
    )
    valid = consecutive & tf.reduce_all(candidate_masks, axis=1)
    starts = tf.boolean_mask(starts, valid)
    window_years = tf.boolean_mask(candidate_years, valid)

    images = tf.map_fn(
        lambda start: record["image"][:, start : start + WINDOW_LENGTH],
        starts,
        fn_output_signature=tf.TensorSpec(
            (len(BAND_INDEXES), WINDOW_LENGTH, PATCH_SIZE, PATCH_SIZE),
            tf.float32,
        ),
    )
    count = tf.shape(starts)[0]
    return tf.data.Dataset.from_tensor_slices(
        {
            "image": images,
            "target": images,
            "years": window_years,
            "temporal_coords": tf.stack(
                [window_years, tf.fill(tf.shape(window_years), DAY_OF_YEAR)],
                axis=-1,
            ),
            "location_coords": tf.repeat(
                record["location_coords"][tf.newaxis, :], count, axis=0
            ),
            "variant": tf.repeat(record["variant"][tf.newaxis], count),
            "patch_id": tf.repeat(record["patch_id"][tf.newaxis], count),
        }
    )


def build_consecutive_window_dataset(paths, shuffle=False, shuffle_buffer=2048):
    paths = [str(path) for path in paths]
    dataset = tf.data.TFRecordDataset(
        paths, compression_type=COMPRESSION, num_parallel_reads=tf.data.AUTOTUNE
    )
    if shuffle:
        dataset = dataset.shuffle(shuffle_buffer, reshuffle_each_iteration=True)
    dataset = dataset.map(
        parse_pretraining_example, num_parallel_calls=tf.data.AUTOTUNE
    )
    dataset = dataset.flat_map(record_to_consecutive_windows)
    if shuffle:
        dataset = dataset.shuffle(shuffle_buffer, reshuffle_each_iteration=True)
    return dataset

## Schema round-trip

A synthetic seven-year record verifies serialization, float16 decoding, and the expected four rolling windows without reading the large source mosaics.

In [11]:
synthetic_years = np.arange(2018, 2025, dtype=np.int64)
synthetic_record = {
    "image": np.zeros(
        (len(BAND_INDEXES), len(synthetic_years), PATCH_SIZE, PATCH_SIZE),
        dtype=STORAGE_DTYPE,
    ),
    "years": synthetic_years,
    "temporal_mask": np.ones(len(synthetic_years), dtype=np.int64),
    "location_coords": np.array([0.0, 0.0], dtype=np.float32),
    "variant": "synthetic",
    "patch_id": "synthetic",
    "patch_row": 0,
    "patch_col": 0,
    "spatial_block": "0:0",
    "split": "train",
    "valid_window_count": 4,
}
synthetic_serialized = serialize_patch(synthetic_record)
synthetic_parsed = parse_pretraining_example(
    tf.convert_to_tensor(synthetic_serialized, dtype=tf.string)
)
synthetic_windows = record_to_consecutive_windows(synthetic_parsed)
synthetic_window_years = [
    item["years"].numpy().tolist() for item in synthetic_windows
]
assert synthetic_parsed["image"].shape == (6, 7, 224, 224)
assert synthetic_window_years == [
    [2018, 2019, 2020, 2021],
    [2019, 2020, 2021, 2022],
    [2020, 2021, 2022, 2023],
    [2021, 2022, 2023, 2024],
]
print("Serialized feature bytes:", f"{len(synthetic_serialized):,}")
print("Rolling windows:", synthetic_window_years)
print("Schema round-trip passed.")

Serialized feature bytes: 4,215,233
Rolling windows: [[2018, 2019, 2020, 2021], [2019, 2020, 2021, 2022], [2020, 2021, 2022, 2023], [2021, 2022, 2023, 2024]]
Schema round-trip passed.


## Streaming sharded export

Records are distributed across shards by a stable hash of `patch_id`. Train and validation writers remain separate. The complete dataset is first written under a `.partial` directory and renamed only after all writers close and split isolation passes, so an interrupted run cannot look complete.

In [12]:
def shard_index(patch_id, shard_count):
    digest = hashlib.sha256(patch_id.encode()).digest()
    return int.from_bytes(digest[:8], "big") % shard_count


def shard_paths(root, split, count):
    folder = root / split
    folder.mkdir(parents=True, exist_ok=False)
    return [
        folder / f"{split}-{index:05d}-of-{count:05d}.tfrecord.gz"
        for index in range(count)
    ]


def prepare_export_root():
    partial_root = OUTPUT_ROOT.with_name(OUTPUT_ROOT.name + ".partial")
    OUTPUT_ROOT.parent.mkdir(parents=True, exist_ok=True)
    if partial_root.exists():
        raise FileExistsError(
            f"Partial export exists: {partial_root}. Inspect or remove it first."
        )
    if OUTPUT_ROOT.exists():
        if not OVERWRITE_OUTPUTS:
            raise FileExistsError(
                f"Output exists: {OUTPUT_ROOT}. Set OVERWRITE_OUTPUTS=True to replace it."
            )
        shutil.rmtree(OUTPUT_ROOT)
    partial_root.mkdir()
    return partial_root


def new_export_stats():
    return {
        "records": Counter(),
        "by_variant": Counter(),
        "rolling_windows": Counter(),
        "skipped_without_valid_window": Counter(),
        "blocks": {"train": set(), "val": set()},
        "started_at": time.time(),
    }


def update_export_stats(stats, record):
    split = record["split"]
    variant = record["variant"]
    stats["records"][split] += 1
    stats["by_variant"][(split, variant)] += 1
    stats["rolling_windows"][(split, variant)] += record["valid_window_count"]
    stats["blocks"][split].add(record["spatial_block"])


def printable_summary(stats, plans_used, output_bytes):
    train_blocks = stats["blocks"]["train"]
    val_blocks = stats["blocks"]["val"]
    return {
        "schema_version": SCHEMA_VERSION,
        "source_root": str(SOURCE_ROOT),
        "output_root": str(OUTPUT_ROOT),
        "elapsed_seconds": time.time() - stats["started_at"],
        "compressed_output_bytes": int(output_bytes),
        "records": dict(stats["records"]),
        "records_by_variant": {
            f"{split}|{variant}": count
            for (split, variant), count in stats["by_variant"].items()
        },
        "rolling_windows_by_variant": {
            f"{split}|{variant}": count
            for (split, variant), count in stats["rolling_windows"].items()
        },
        "skipped_without_valid_window": dict(
            stats["skipped_without_valid_window"]
        ),
        "train_spatial_blocks": len(train_blocks),
        "val_spatial_blocks": len(val_blocks),
        "spatial_block_overlap": len(train_blocks & val_blocks),
        "variants": {plan.variant: list(plan.years) for plan in plans_used},
        "configuration": {
            "min_year": MIN_YEAR,
            "max_year": MAX_YEAR,
            "window_length": WINDOW_LENGTH,
            "patch_size": PATCH_SIZE,
            "stride": STRIDE,
            "band_indexes": list(BAND_INDEXES),
            "storage_dtype": STORAGE_DTYPE_NAME,
            "compression": COMPRESSION,
            "val_fraction": VAL_FRACTION,
            "split_seed": SPLIT_SEED,
            "spatial_block_degrees": SPATIAL_BLOCK_DEGREES,
            "train_shards": TRAIN_SHARDS,
            "val_shards": VAL_SHARDS,
            "include_edge_patches": INCLUDE_EDGE_PATCHES,
            "skip_without_valid_window": SKIP_WITHOUT_VALID_WINDOW,
            "max_variants": MAX_VARIANTS,
            "max_stripes_per_variant": MAX_STRIPES_PER_VARIANT,
            "max_patches_per_stripe": MAX_PATCHES_PER_STRIPE,
        },
    }

In [13]:
def export_tfrecords(plans):
    if audit_issues:
        raise RuntimeError("Resolve source audit issues before exporting")

    plans_used = plans[:MAX_VARIANTS] if MAX_VARIANTS else plans
    partial_root = prepare_export_root()
    train_paths = shard_paths(partial_root, "train", TRAIN_SHARDS)
    val_paths = shard_paths(partial_root, "val", VAL_SHARDS)
    options = tf.io.TFRecordOptions(compression_type=COMPRESSION)
    stats = new_export_stats()
    processed = 0

    with ExitStack() as writer_stack:
        writers = {
            "train": [
                writer_stack.enter_context(
                    tf.io.TFRecordWriter(str(path), options=options)
                )
                for path in train_paths
            ],
            "val": [
                writer_stack.enter_context(
                    tf.io.TFRecordWriter(str(path), options=options)
                )
                for path in val_paths
            ],
        }

        for plan_number, plan in enumerate(plans_used, start=1):
            print(
                f"\n[{plan_number}/{len(plans_used)}] {plan.variant} | years={plan.years}"
            )
            with open_plan_sources(plan) as sources:
                stripe_number = 0
                for row in range(0, REFERENCE_HEIGHT, STRIDE):
                    if (
                        MAX_STRIPES_PER_VARIANT is not None
                        and stripe_number >= MAX_STRIPES_PER_VARIANT
                    ):
                        break
                    if (
                        not INCLUDE_EDGE_PATCHES
                        and REFERENCE_HEIGHT - row < PATCH_SIZE
                    ):
                        break
                    stripe = read_temporal_stripe(sources, row)
                    written_in_stripe = 0

                    for col in range(0, REFERENCE_WIDTH, STRIDE):
                        if (
                            not INCLUDE_EDGE_PATCHES
                            and REFERENCE_WIDTH - col < PATCH_SIZE
                        ):
                            break
                        if (
                            MAX_PATCHES_PER_STRIPE is not None
                            and written_in_stripe >= MAX_PATCHES_PER_STRIPE
                        ):
                            break
                        record = extract_patch_from_stripe(
                            plan, stripe, row, col
                        )
                        if (
                            SKIP_WITHOUT_VALID_WINDOW
                            and record["valid_window_count"] == 0
                        ):
                            stats["skipped_without_valid_window"][
                                plan.variant
                            ] += 1
                            continue

                        split = record["split"]
                        shard_count = TRAIN_SHARDS if split == "train" else VAL_SHARDS
                        index = shard_index(record["patch_id"], shard_count)
                        writers[split][index].write(serialize_patch(record))
                        update_export_stats(stats, record)
                        written_in_stripe += 1
                        processed += 1

                        if processed % PROGRESS_EVERY == 0:
                            elapsed = time.time() - stats["started_at"]
                            rate = processed / max(elapsed, 1e-9)
                            print(
                                f"  records={processed:,} | "
                                f"train={stats['records']['train']:,} | "
                                f"val={stats['records']['val']:,} | "
                                f"{rate:.2f} records/s"
                            )

                    del stripe
                    stripe_number += 1
                    print(
                        f"  stripe {stripe_number}: row={row:,}, "
                        f"wrote={written_in_stripe:,}"
                    )

    train_blocks = stats["blocks"]["train"]
    val_blocks = stats["blocks"]["val"]
    if not train_blocks.isdisjoint(val_blocks):
        raise RuntimeError("Geographic train/validation leakage detected")

    output_bytes = sum(path.stat().st_size for path in partial_root.rglob("*.gz"))
    summary = printable_summary(stats, plans_used, output_bytes)
    (partial_root / "summary.json").write_text(
        json.dumps(summary, indent=2), encoding="utf-8"
    )
    os.replace(partial_root, OUTPUT_ROOT)
    print("\nExport complete:", OUTPUT_ROOT)
    print(json.dumps(summary, indent=2))
    return summary

In [5]:
RUN_EXPORT = True

In [ ]:
if RUN_EXPORT:
    export_summary = export_tfrecords(plans)
else:
    export_summary = None
    print("Export skipped because RUN_EXPORT=False.")
    print("Review the inventory, audit, split, and size estimate first.")


[1/4] harmonized | years=(2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024)
  stripe 1: row=0, wrote=108
  stripe 2: row=224, wrote=108
  stripe 3: row=448, wrote=108
  stripe 4: row=672, wrote=108
  records=500 | train=437 | val=63 | 0.50 records/s
  stripe 5: row=896, wrote=108
  stripe 6: row=1,120, wrote=108
  stripe 7: row=1,344, wrote=108
  stripe 8: row=1,568, wrote=108
  stripe 9: row=1,792, wrote=108
  records=1,000 | train=872 | val=128 | 0.50 records/s
  stripe 10: row=2,016, wrote=108
  stripe 11: row=2,240, wrote=108
  stripe 12: row=2,464, wrote=153
  stripe 13: row=2,688, wrote=153
  records=1,500 | train=1,314 | val=186 | 0.53 records/s
  stripe 14: row=2,912, wrote=153
  stripe 15: row=3,136, wrote=153
  stripe 16: row=3,360, wrote=153
  records=2,000 | train=1,708 | val=292 | 0.57 records/s
  stripe 17: row=3,584, wrote=153
  stripe 18: row=3,808, wrote=153
  stripe 19: row=4,032, wrote=153
  records=2,500 | train=2,112 | val=388 | 0.60 records/s
  stripe 2

PermissionError: [WinError 5] Access is denied: 'D:\\Data\\ssm_temporal\\prithvi_continous_pretraining.partial' -> 'D:\\Data\\ssm_temporal\\prithvi_continous_pretraining'

: 

## Post-export verification

This samples one example from every nonempty shard, checks schema metadata and tensor shape, and verifies that all expanded windows contain four consecutive years. Full record counts are preserved in `summary.json` during export.

In [8]:
def verify_export(output_root=OUTPUT_ROOT):
    summary_path = output_root / "summary.json"
    if not summary_path.exists():
        raise FileNotFoundError(summary_path)
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    assert summary["schema_version"] == SCHEMA_VERSION
    assert summary["spatial_block_overlap"] == 0

    checked = Counter()
    for split in ("train", "val"):
        for path in sorted((output_root / split).glob("*.tfrecord.gz")):
            if path.stat().st_size == 0:
                continue
            dataset = tf.data.TFRecordDataset(
                [str(path)], compression_type=COMPRESSION
            ).take(1)
            for serialized in dataset:
                parsed = parse_pretraining_example(serialized)
                assert parsed["image"].shape[0] == len(BAND_INDEXES)
                assert parsed["image"].shape[2:] == (PATCH_SIZE, PATCH_SIZE)
                windows = record_to_consecutive_windows(parsed)
                for item in windows.take(1):
                    years = item["years"].numpy()
                    assert len(years) == WINDOW_LENGTH
                    assert np.all(np.diff(years) == 1)
                    assert item["image"].shape == (6, 4, 224, 224)
                checked[split] += 1
    print("Verified nonempty shards:", dict(checked))
    print("Post-export verification passed.")
    return summary


if RUN_EXPORT and OUTPUT_ROOT.exists():
    verified_summary = verify_export()
else:
    verified_summary = None
    print("Run the export before post-export verification.")

NameError: name 'FEATURE_DESCRIPTION' is not defined

## Final checklist

- Confirm the discovered years and rolling windows for every product.
- Resolve every source-audit issue before export.
- Check the geographic split counts and disk estimate.
- Optionally enable `RUN_PREVIEW` and inspect a representative patch.
- Confirm the output path has enough free space.
- Set `RUN_EXPORT=True` only when ready for the full run.
- Keep `summary.json` with the TFRecord shards.
- In the training notebook, apply normalization and random MAE masking after four-year expansion; keep the original tensor as the reconstruction target.